Goal: Preprocess the data, including training sentences, tokenizing and aligning labels

In [12]:
import torch
import transformers
from transformers import BertTokenizerFast, BertForTokenClassification
from TorchCRF import CRF
from torch.utils.data import DataLoader, Dataset
import numpy as np

In [44]:
label_map = {
    "O": 0,
    "B-PROPERTY": 1, #主语
    "I-PROPERTY": 2,
    "B-SUBPROPERTY": 3,
    "I-SUBPROPERTY": 4,
    "B-YEAR": 5, #year
    "I-YEAR": 6, 
    "B-TIMESTAMP": 7,
    "I-TIMESTAMP": 8,
    "B-UNIT": 9, #currency, dollar sign, etc. optional, some value may not have a unit
    "I-UNIT": 10,
    "B-VALUE": 11, #number only
    "I-VALUE": 12, 
    "B-MULTIPLIER": 13,
    "I-MULTIPLIER": 14,
    "B-COMPANY": 15, #if we want to compare different companies
    "I-COMPANY": 16,
    "B-CHANGE-UP": 17, #increase, decrease, to be implemented
    "I-CHANGE-IP": 18,
    "B-CHANGE-DOWN": 19,
    "I-CHANGE-DOWN": 20,
    "B-RATIO": 21, #to be implemented
    "I-RATIO": 22
}

In [35]:
train_sentences = [
    "GMV in 2023 was $ 19.2 billion",
    "Revenue in 2021 was $ 4.38 billion",
    "Revenue in 2022 was $ 5.56 billion",
    "Revenue in 2023 was $ 7.68 billion",
    "Profit in 2021 was $ 24 million",
    "Profit in 2022 was $ 372 million",
    "Profit in 2023 was $ 1.10 billion",
    "Adjusted profit in 2021 was $ 769.6 million",
    "Adjusted profit in 2022 was $ 788.1 million",
    "Adjusted profit in 2023 was $ 1.46 billion"
]

train_labels = [
    ["B-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-SUBPROPERTY", "B-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-SUBPROPERTY", "B-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-SUBPROPERTY", "B-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"]
]

train_sentences.extend([
    "Tesla 's revenue in 2023 was $ 81.5 billion",
    "Amazon 's net profit in 2022 was $ 33.3 billion",
    "Apple 's operating income in 2024 was $ 34.2 billion",
    "Google 's advertising revenue in 2023 was $ 280 billion",
    "Microsoft 's cloud revenue in 2022 was $ 72 billion",
    "Facebook 's profit margin in 2023 was 25 %",
    "Netflix 's subscriber growth in 2023 was 12 %",
    "Goldman Sachs 's investment banking revenue in 2021 was $ 14.5 billion",
    "JP Morgan 's total assets in 2023 were $ 3.9 trillion",
    "Berkshire Hathaway 's total revenue in 2023 was $ 302 billion"
])

train_labels.extend([
    ["B-COMPANY", "O", "B-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-COMPANY", "O", "B-SUBPROPERTY", "B-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-COMPANY", "O", "B-PROPERTY", "I-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-COMPANY", "O", "B-SUBPROPERTY", "B-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-COMPANY", "O", "B-SUBPROPERTY", "B-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-COMPANY", "O", "B-PROPERTY", "I-PROPERTY", "O", "B-YEAR", "O", "B-VALUE", "B-UNIT"],
    ["B-COMPANY", "O", "B-PROPERTY", "I-PROPERTY", "O", "B-YEAR", "O", "B-VALUE", "B-UNIT"],
    ["B-COMPANY", "I-COMPANY", "O", "B-SUBPROPERTY", "I-SUBPROPERTY", "B-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-COMPANY", "I-COMPANY", "O", "B-SUBPROPERTY", "B-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"],
    ["B-COMPANY", "I-COMPANY", "O", "B-SUBPROPERTY", "B-PROPERTY", "O", "B-YEAR", "O", "B-UNIT", "B-VALUE", "B-MULTIPLIER"]
])

train_sentences.extend([
    "Goodme 's profit for the year was 24.0 million dollars , 372.0 million dollars and 1,096.4 million dollars in 2021 , 2022 and 2023 , respectively.",
    "Goodme 's adjusted profit was 769.6 million dollars , 788.1 million dollars and 1,459.0 million dollars in 2021 , 2022 and 2023 , respectively.",
    "The revenue in 2021 was 4.38 billion dollars",
    "The revenue in 2022 was 5.56 billion dollars",
    "The revenue in 2023 was 7.68 billion dollars",
    "Profit in 2021 was 24 million dollars",
    "Profit in 2022 was 372 million dollars",
    "Profit in 2023 was 1.10 billion dollars",
    "Adjusted profit in 2021 was 769.6 million dollars",
    "Adjusted profit in 2022 was 788.1 million dollars",
    "Adjusted profit in 2023 was 1.46 billion dollars"
])

train_labels.extend([
    ["B-COMPANY", "O", "B-PROPERTY", "O", "O", "O", "O", "B-VALUE", "B-MULTIPLIER", "B-UNIT", "O", "B-VALUE", "B-MULTIPLIER", "B-UNIT", "O","B-VALUE", "B-MULTIPLIER", "B-UNIT", "O", "B-YEAR", "O", "B-YEAR", "O", "B-YEAR", "O", "O", "O"],
    ["B-COMPANY", "O", "B-SUBPROPERTY", "B-PROPERTY",  "O", "B-VALUE", "B-MULTIPLIER", "B-UNIT", "O", "B-VALUE", "B-MULTIPLIER", "B-UNIT", "O","B-VALUE", "B-MULTIPLIER", "B-UNIT", "O", "B-YEAR", "O", "B-YEAR", "O", "B-YEAR", "O", "O", "O"],
    ["O", "B-PROPERTY", "O", "B-YEAR", "O", "B-VALUE", "B-MULTIPLIER", "B-UNIT"],
    ["O", "B-PROPERTY", "O", "B-YEAR", "O", "B-VALUE", "B-MULTIPLIER", "B-UNIT"],
    ["O", "B-PROPERTY", "O", "B-YEAR", "O", "B-VALUE", "B-MULTIPLIER", "B-UNIT"],
    ["B-PROPERTY", "O", "B-YEAR", "O", "B-VALUE", "B-MULTIPLIER", "B-UNIT"],
    ["B-PROPERTY", "O", "B-YEAR", "O", "B-VALUE", "B-MULTIPLIER", "B-UNIT"],
    ["B-PROPERTY", "O", "B-YEAR", "O", "B-VALUE", "B-MULTIPLIER", "B-UNIT"],
    ["B-SUBPROPERTY", "B-PROPERTY", "O", "B-YEAR", "O", "B-VALUE", "B-MULTIPLIER", "B-UNIT"],
    ["B-SUBPROPERTY", "B-PROPERTY", "O", "B-YEAR", "O", "B-VALUE", "B-MULTIPLIER", "B-UNIT"],
    ["B-SUBPROPERTY", "B-PROPERTY", "O", "B-YEAR", "O", "B-VALUE", "B-MULTIPLIER", "B-UNIT"]
])

train_sentences.extend([
    "Our other expenses amounted to 5.8 million , 1.1 million and 9.5 million dollars for the years ended December 31 , 2021 , 2022 and 2023 , respectively .",
    "Our finance costs amounted to 5.1 million , 5.4 million and 5.2 million dollars for the years ended December 31 , 2021 , 2022 and 2023 , respectively , and amounted to 4.3 million and 2.2 million dollar in the nine months ended September 30 , 2023 and 2024 , respectively .",
    "Revenue grew by 5 % .",
    "Cost decreased by 7 % .",
    "Revenue increased by 8 % .",
    "Revenue showed growth of 7 % ."
])

train_labels.extend([
    ["O", "B-SUBPROPERTY", "B-PROPERTY", "O", "O", "B-VALUE", "B-MULTIPLIER", "O","B-VALUE", "B-MULTIPLIER", "O","B-VALUE", "B-MULTIPLIER", "B-UNIT", "O", "O", "B-TIMESTAMP", "I-TIMESTAMP", "I-TIMESTAMP", "I-TIMESTAMP", "O", "B-YEAR", "O", "B-YEAR", "O", "B-YEAR", "O", "O", "O"],
    ["O", "B-SUBPROPERTY", "B-PROPERTY", "O", "O", "B-VALUE", "B-MULTIPLIER", "O", "B-VALUE", "B-MULTIPLIER", "O", "B-VALUE", "B-MULTIPLIER", "B-UNIT", "O", "O", "B-TIMESTAMP", "I-TIMESTAMP", "I-TIMESTAMP", "I-TIMESTAMP", "O", "B-YEAR", "O", "B-YEAR", "O", "B-YEAR", "O", "O", "O", "O", "O", "O", "B-VALUE", "B-MULTIPLIER", "O", "B-VALUE", "B-MULTIPLIER", "B-UNIT", "O", "O", "B-TIMESTAMP", "I-TIMESTAMP", "I-TIMESTAMP", "I-TIMESTAMP", "I-TIMESTAMP", "O", "B-YEAR", "O", "B-YEAR", "O", "O", "O"],
    ["B-PROPERTY", "O", "O", "B-CHANGE-UP", "I-CHANGE-UP", "O"],
    ["B-PROPERTY", "O", "O", "B-CHANGE-DOWN", "I-CHANGE-DOWN", "O"],
    ["B-PROPERTY", "O", "O", "B-CHANGE-UP", "I-CHANGE-UP", "O"],
    ["B-PROPERTY", "O", "O", "O", "B-CHANGE-UP", "I-CHANGE-UP", "O"]
])

Preprocess:
aligns tokenized words with assigned labels 

In [31]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

In [32]:
def align_labels(sentence, word_labels):
    words = sentence.split()
    tokens = []
    aligned_labels = []

    word_idx = 0  # of given label list

    for word in words:
        sub_tokens = tokenizer.tokenize(word)
        tokens.extend(sub_tokens)

        first_label = word_labels[word_idx]
        if first_label.startswith("B-"):
            sub_labels = [first_label] + ["I-" + first_label[2:]] * (len(sub_tokens) - 1)
        else: #"I-" or "O"
            sub_labels = [first_label] + [first_label] * (len(sub_tokens) - 1)

        aligned_labels.extend(sub_labels)
        word_idx += 1

    return tokens, aligned_labels

In [45]:
def tokenize_and_align_labels(sentences, labels, max_length=100):
    tokenized_inputs = {"input_ids": [], "attention_mask": [], "labels": []}

    for i, (sentence, word_labels) in enumerate(zip(sentences, labels)):
        tokens, aligned_labels = align_labels(sentence, word_labels)

        # adding [CLS] and [SEP], don't see the use yet, but seems a common protocol
        tokens = ["[CLS]"] + tokens + ["[SEP]"]
        aligned_labels = ["O"] + aligned_labels + ["O"] 

        # Convert tokens to input IDs
        input_ids = tokenizer.convert_tokens_to_ids(tokens)
        attention_mask = [1] * len(input_ids)

        # Convert aligned labels to numerical values using `label_map`
        label_ids = [label_map.get(lbl, 0) for lbl in aligned_labels]

        # Pad sequences to max_length
        padding_length = max_length - len(input_ids)
        if padding_length > 0:
            input_ids += [0] * padding_length  # Pad input IDs with 0 (BERT's padding)
            attention_mask += [0] * padding_length  # Pad attention mask with 0
            label_ids += [-100] * padding_length  # Pad labels with -100 to ignore them in loss

        tokenized_inputs["input_ids"].append(input_ids)
        tokenized_inputs["attention_mask"].append(attention_mask)
        tokenized_inputs["labels"].append(label_ids)
    return tokenized_inputs

In [46]:
for i, sentence in enumerate(train_sentences):
    tokens, adjusted_labels = align_labels(sentence, train_labels[i])
    print(f"Sentence {i}: {sentence}")
    print(f"Tokenized: {tokens}")
    print(f"Aligned Labels: {adjusted_labels}")
    print("-" * 40)

Sentence 0: GMV in 2023 was $ 19.2 billion
Tokenized: ['gm', '##v', 'in', '202', '##3', 'was', '$', '19', '.', '2', 'billion']
Aligned Labels: ['B-PROPERTY', 'I-PROPERTY', 'O', 'B-YEAR', 'I-YEAR', 'O', 'B-UNIT', 'B-VALUE', 'I-VALUE', 'I-VALUE', 'B-MULTIPLIER']
----------------------------------------
Sentence 1: Revenue in 2021 was $ 4.38 billion
Tokenized: ['revenue', 'in', '2021', 'was', '$', '4', '.', '38', 'billion']
Aligned Labels: ['B-PROPERTY', 'O', 'B-YEAR', 'O', 'B-UNIT', 'B-VALUE', 'I-VALUE', 'I-VALUE', 'B-MULTIPLIER']
----------------------------------------
Sentence 2: Revenue in 2022 was $ 5.56 billion
Tokenized: ['revenue', 'in', '202', '##2', 'was', '$', '5', '.', '56', 'billion']
Aligned Labels: ['B-PROPERTY', 'O', 'B-YEAR', 'I-YEAR', 'O', 'B-UNIT', 'B-VALUE', 'I-VALUE', 'I-VALUE', 'B-MULTIPLIER']
----------------------------------------
Sentence 3: Revenue in 2023 was $ 7.68 billion
Tokenized: ['revenue', 'in', '202', '##3', 'was', '$', '7', '.', '68', 'billion']
Align

In [47]:
train_encodings = tokenize_and_align_labels(train_sentences, train_labels)

In [48]:
class NERDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        return {key: torch.tensor(val[idx]) for key, val in self.encodings.items()} 

In [49]:
train_dataset = NERDataset(train_encodings)
train_dataloader = DataLoader(train_dataset, batch_size=8, shuffle=True)

In [50]:
class BertCRF(torch.nn.Module):
    def __init__(self, num_labels):
        super(BertCRF, self).__init__()
        self.bert = BertForTokenClassification.from_pretrained("bert-base-uncased", num_labels=num_labels)
        self.crf = CRF(num_labels, batch_first=True)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids, attention_mask=attention_mask, return_dict=False)
        emissions = outputs[0]

        if labels is not None:
            labels = labels.clone()  # Avoid modifying original tensor
            labels[labels == -100] = 0  # Replace -100 with 'O'

            # Compute CRF loss
            loss = -self.crf(emissions, labels, mask=attention_mask.bool(), reduction="mean")
            return loss
        else:
            return self.crf.decode(emissions, mask=attention_mask.bool())


In [51]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BertCRF(num_labels=len(label_map)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [52]:
def train_model(num_epochs=3):
    model.train()
    for epoch in range(num_epochs):
        total_loss = 0
        for batch in train_dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            loss = model(input_ids, attention_mask, labels)
            total_loss += loss.item()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        print(f"Epoch {epoch+1} Loss: {total_loss / len(train_dataloader):.4f}")

In [53]:
train_model(num_epochs=10)

Epoch 1 Loss: 41.5603
Epoch 2 Loss: 21.0621
Epoch 3 Loss: 11.8344
Epoch 4 Loss: 7.3176
Epoch 5 Loss: 5.0952
Epoch 6 Loss: 3.6162
Epoch 7 Loss: 2.5402
Epoch 8 Loss: 1.9215
Epoch 9 Loss: 1.4168
Epoch 10 Loss: 1.0606


In [54]:
def save_model(model, tokenizer, label_map, save_directory="bert_crf_model"):
    """Saves the trained model, tokenizer, and label map."""
    import os
    os.makedirs(save_directory, exist_ok=True)

    model_path = f"{save_directory}/bert_crf_model.pt"
    torch.save(model.state_dict(), model_path)
    print(f"model weight successfully saved to {model_path}")

    tokenizer.save_pretrained(save_directory)
    print(f"tokenizer successfully saved to {save_directory}")

    label_map_path = f"{save_directory}/label_map.json"
    import json
    with open(label_map_path, "w") as f:
        json.dump(label_map, f)
    print(f"label map successfully saved to {label_map_path}")

save_model(model, tokenizer, label_map)

model weight successfully saved to bert_crf_model/bert_crf_model.pt
tokenizer successfully saved to bert_crf_model
label map successfully saved to bert_crf_model/label_map.json
